# Daily-Reversal-Strategie bauen und testen

Dieses Notebook baut eine einfache Daily-Reversal-Strategie nach und stresst sie danach mit den Mindestfragen, die ein Trading-Backtest beantworten muss: Benchmark, Drawdown, Kosten, Out-of-Sample und Cross-Asset-Robustheit.

Die Idee ist bewusst einfach:

- Wenn der gestrige Log Return negativ war, ist das heutige Signal long.
- Wenn der gestrige Log Return positiv war, ist das heutige Signal short.
- Wenn kein gültiger gestriger Return existiert, wird keine Position eröffnet.

Das Notebook ist kein Trading-Signal und keine Anlageberatung. Es ist ein Research-Beispiel: Eine einfache Hypothese wird in Code übersetzt und dann Schritt für Schritt härter geprüft.

## Notebook-Aufbau

1. Setup und editierbare Parameter
2. Daten laden und Datenqualität prüfen
3. Log Returns sauber berechnen
4. Daily-Reversal-Signal und Strategy Returns
5. Equity Curve vs Buy-and-Hold
6. Performance-Metriken, Drawdown und Calmar Ratio
7. Kostenstress und Turnover
8. In-Sample / Out-of-Sample
9. Jahresperformance als Regime-Check
10. Cross-Asset-Vergleich
11. Fazit und nächste Research-Fragen

## 1. Setup und editierbare Parameter

Diese Zelle enthält die zentralen Annahmen. Hier können Assets, Startdatum, Startkapital und Annualisierungsannahme geändert werden, ohne die spätere Logik anzufassen.

In [ ]:
# 1. Setup
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

pd.options.display.float_format = "{:,.4f}".format

ASSETS = {
    "Ethereum": "ETH-USD",
    "Gold": "GLD",
    "EUR/USD": "EURUSD=X",
    "S&P 500": "SPY",
}

START_DATE = "2018-01-01"
END_DATE = None

START_CAPITAL = 10000
TRADING_DAYS = 252

BUY_HOLD_NAME = "Buy & Hold"
STRATEGY_NAME = "Daily Reversal"

## 2. Daten laden und Datenqualität prüfen

Wir nutzen Yahoo-Finance-Daten über `yfinance`. Die Datenquelle ist praktisch und kostenlos, aber nicht perfekt. Deshalb prüfen wir direkt, ab wann jedes Asset Daten hat und wie viele fehlende Werte entstehen.

In [ ]:
# 2. Daten laden
tickers = list(ASSETS.values())

raw_data = yf.download(
    tickers=tickers,
    start= START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=False
)

raw_data.head()

In [ ]:
# Closing prices extrahieren
prices = raw_data["Close"].copy()

prices = prices.rename(columns={ticker: name for name, ticker in ASSETS.items()})

prices.tail(8)

In [ ]:
# Datenqualität prüfen

data_quality = pd.DataFrame({
    "first_date": prices.apply(lambda x: x.first_valid_index()),
    "last_date": prices.apply(lambda x: x.last_valid_index()),
    "observations": prices.notna().sum(),
    "missing_values": prices.isna().sum(),
})

data_quality

## 3. Log Returns sauber berechnen

Die Assets handeln auf unterschiedlichen Kalendern. Deshalb werden die Returns pro Asset nach `dropna()` berechnet, statt naive Returns über den ganzen gemeinsamen DataFrame zu ziehen.

In [ ]:
# Log Returns sauber berechenen
def calculate_log_returns(price_series):
    price_series = price_series.dropna()
    return np.log(price_series / price_series.shift(1))

log_returns = pd.concat(
    {
        asset: calculate_log_returns(prices[asset])
        for asset in prices.columns
    },
    axis=1
)

log_returns.tail(10)

In [ ]:
#Return-Datenqualität prüfen
return_quality = pd.DataFrame({
    "first_return": log_returns.apply(lambda x: x.first_valid_index()),
    "last_return": log_returns.apply(lambda x: x.last_valid_index()),
    "return_observations": log_returns.notna().sum(),
    "missing_returns": log_returns.isna().sum(),
})

return_quality

## 4. Daily-Reversal-Signal und Strategy Returns

Das Signal für heute basiert nur auf dem gestrigen Log Return. Dadurch entsteht kein offensichtlicher Lookahead Bias in der Signalbildung.

In [ ]:
#Die Regel: 
#Wenn der gestrige Return negativ war -> heute long
#Wenn der gestrige Return positiv war -> heute short
#Wenn es keinen gestrigen Return gibt -> keine Position
asset = "Ethereum"

asset_returns = log_returns[asset].dropna()

signal = -np.sign(asset_returns.shift(1))
signal = signal.fillna(0)

signal.tail(10)

In [ ]:
# Signal und Returns zusammen anschauen
signal_check = pd.DataFrame({
    "close_log_ret": asset_returns,
    "close_log_ret_lag_1": asset_returns.shift(1),
    "reversal_signal": signal,
})

signal_check.tail(10)

In [ ]:
strategy_returns = signal * asset_returns

strategy_check = pd.DataFrame({
    "close_log_ret": asset_returns,
    "close_log_ret_lag_1": asset_returns.shift(1),
    "reversal_signal": signal,
    "strategy_log_ret": strategy_returns,
})

strategy_check.tail(10)

## 5. Equity Curve vs Buy-and-Hold

Jetzt vergleichen wir die Strategie gegen den naheliegenden Benchmark: das Asset einfach kaufen und halten. Die Equity Curve wird korrekt aus kumulierten Log Returns mit `exp(cumsum(...))` berechnet.

In [ ]:
strategy_cum_log_ret = strategy_returns.cumsum()
buy_hold_cum_log_ret = asset_returns.cumsum()

performance = pd.DataFrame({
    BUY_HOLD_NAME: buy_hold_cum_log_ret,
    STRATEGY_NAME: strategy_cum_log_ret,
})

performance.tail()

In [ ]:
#Equity Curves
equity_curves = START_CAPITAL * np.exp(performance)

equity_curves.tail()

In [ ]:
equity_curves.plot(figsize=(12, 6), title=f"{asset}: Buy-and-Hold vs Daily Reversal")
plt.ylabel("Portfolio Value")
plt.xlabel("Date")
plt.grid(True)
plt.show()

In [ ]:
# Total Return vergleichen
total_return = equity_curves.iloc[-1] / START_CAPITAL - 1

In [ ]:
total_return_table = pd.DataFrame({
    "total_return": total_return,
    "total_return_%": total_return*100,
    "final_value": equity_curves.iloc[-1]
})

total_return_table.index = ["Buy & Hold", "Daily Reversal"]
total_return_table

total_return_table

## 6. Performance-Metriken, Drawdown und Calmar Ratio

Eine Strategie ist nicht nur eine Endkapitalzahl. Wir prüfen Trefferquote, Volatilität, Sharpe, Sortino, Drawdown und Calmar Ratio.

In [ ]:
# Performance Metriken vorbereiten
comparison_returns = pd.DataFrame({
    BUY_HOLD_NAME: asset_returns,
    STRATEGY_NAME: strategy_returns,
})

comparison_returns.tail()

In [ ]:
metrics = pd.DataFrame({
    "mean_daily_return": comparison_returns.mean(),
    "volatility_daily": comparison_returns.std(),
    "win_rate": (comparison_returns > 0).mean(),
    "best_day": comparison_returns.max(),
    "worst_day": comparison_returns.min(),
})

metrics

<!-- metric-formula-cells-v1 -->
### Sharpe Ratio

Die Sharpe Ratio misst den Excess Return pro Einheit Gesamtrisiko:

$$
\text{Sharpe Ratio} =
\frac{R_p - R_f}{\sigma_p}
$$

Für Tagesdaten annualisieren wir die vereinfachte Form im Notebook so:

$$
\text{Sharpe Ratio}_{\text{ann.}} =
\frac{\bar{r}_p - \bar{r}_f}{\sigma_p} \cdot \sqrt{N}
$$

\(R_p\) ist die Portfoliorendite, \(R_f\) die risikofreie Rendite und \(\sigma_p\) die Standardabweichung der Portfoliorenditen. Im Notebook setzen wir \(R_f = 0\), damit die Berechnung für das Tutorial schlank bleibt.

In [ ]:
# Sharpe Ratio
sharpe_ratio = (
    comparison_returns.mean()
    / comparison_returns.std()
    * np.sqrt(TRADING_DAYS)
)

sharpe_ratio

<!-- metric-formula-cells-v1 -->
### Sortino Ratio

Die Sortino Ratio misst den Excess Return pro Einheit Downside-Risiko:

$$
\text{Sortino Ratio} =
\frac{R_p - R_T}{\sigma_d}
$$

$$
\sigma_d =
\sqrt{\frac{1}{N}\sum_{t=1}^{N}\min(0, R_{p,t} - R_T)^2}
$$

\(R_T\) ist die Zielrendite oder Minimum Acceptable Return. In vielen Anwendungen wird dafür \(0\) oder die risikofreie Rendite genutzt. Im Notebook setzen wir \(R_T = 0\) und annualisieren analog zur Sharpe Ratio mit \(\sqrt{N}\).

In [ ]:
# Sortino Ratio
# Downside deviation gemaess Formel: sqrt(1/N * sum(min(0, R_t - R_T)^2)) mit R_T = 0
downside_part = comparison_returns.clip(upper=0)
downside_deviation = np.sqrt((downside_part ** 2).mean())

sortino_ratio = (
    comparison_returns.mean()
    / downside_deviation
    * np.sqrt(TRADING_DAYS)
)

sortino_ratio

In [ ]:
metrics["sharpe_ratio"] = sharpe_ratio
metrics["sortino_ratio"] = sortino_ratio

metrics

In [ ]:
# Drawdown berechnen
running_max = equity_curves.cummax()
drawdowns = equity_curves / running_max - 1

drawdowns.tail()

In [ ]:
max_drawdown = drawdowns.min()
max_drawdown

In [ ]:
drawdowns.plot(figsize=(12, 5), title=f"{asset}: Drawdowns")
plt.ylabel("Drawdown")
plt.xlabel("Date")
plt.grid(True)
plt.show()

<!-- metric-formula-cells-v1 -->
### CAGR

Die CAGR ist die annualisierte Wachstumsrate über den gesamten Zeitraum:

$$
\text{CAGR} =
\left(\frac{V_{\text{end}}}{V_{\text{begin}}}\right)^{\frac{1}{Y}} - 1
$$

\(V_{\text{begin}}\) ist der Anfangswert, \(V_{\text{end}}\) der Endwert und \(Y\) die Länge des Tests in Jahren. Im Code ist das äquivalent zur Berechnung über kumulierte Log Returns.

In [ ]:
# CAGR berechnen
years = (comparison_returns.index[-1] - comparison_returns.index[0]).days / 365.25

cagr = np.exp(comparison_returns.sum()) ** (1 / years) - 1

cagr

<!-- metric-formula-cells-v1 -->
### Calmar Ratio

Die Calmar Ratio setzt annualisierte Rendite ins Verhältnis zum maximalen Drawdown:

$$
\text{Calmar Ratio} =
\frac{\text{Annualized Return}}{\left|\text{Maximum Drawdown}\right|}
$$

Im Notebook verwenden wir für die annualisierte Rendite die CAGR. Der maximale Drawdown steht im DataFrame als negativer Wert, deshalb nutzen wir im Nenner den Betrag.

In [ ]:
# Calmar Ratio
calmar_ratio = cagr / abs(max_drawdown)

calmar_ratio

In [ ]:
metrics["cagr"] = cagr
metrics["max_drawdown"] = max_drawdown
metrics["calmar_ratio"] = calmar_ratio

metrics

## 7. Kostenstress und Turnover

Daily-Reversal kann viele Positionswechsel erzeugen. Deshalb ist Turnover zentral: Kosten entstehen nicht abstrakt, sondern durch Positionsänderungen.

In [ ]:
# Turnover berechnen
turnover = signal.diff().abs()
turnover.iloc[0] = abs(signal.iloc[0])

turnover.tail()

In [ ]:
# Kostenmodell
COST_BPS = 10

costs = turnover * (COST_BPS / 10000)

strategy_log_ret_after_costs = strategy_returns - costs

In [ ]:
strategy_equity_after_costs = START_CAPITAL * np.exp(strategy_log_ret_after_costs.cumsum())

equity_with_costs = equity_curves.copy()
equity_with_costs[f"{STRATEGY_NAME} ({COST_BPS} bps costs)"] = strategy_equity_after_costs

equity_with_costs.tail()

In [ ]:
turnover_summary = pd.Series({
    "average_daily_turnover": turnover.mean(),
    "total_turnover": turnover.sum(),
    "number_of_position_changes": (turnover > 0).sum(),
    "number_of_observations": turnover.count(),
})

turnover_summary

In [ ]:
total_cost_drag = costs.sum()

cost_impact = pd.Series({
    "total_cost_drag_log_return": total_cost_drag,
    "gross_strategy_cum_log_return": strategy_returns.cumsum().iloc[-1],
    "net_strategy_cum_log_return": strategy_log_ret_after_costs.cumsum().iloc[-1],
})

cost_impact

In [ ]:
equity_with_costs.plot(
    figsize=(12, 6),
    title=f"{asset}: Buy-and-Hold vs Daily Reversal Before and After Costs"
)

plt.ylabel("Portfolio Value")
plt.xlabel("Date")
plt.grid(True)
plt.show()

In [ ]:
# Returns mit Kosten vergleichen
comparison_returns_with_costs = pd.DataFrame({
    BUY_HOLD_NAME: asset_returns,
    STRATEGY_NAME: strategy_returns,
    f"{STRATEGY_NAME} ({COST_BPS} bps costs)": strategy_log_ret_after_costs,
})

comparison_returns_with_costs.tail()

In [ ]:
def calculate_metrics(return_df, equity_df):
    metrics = pd.DataFrame({
        "mean_daily_return": return_df.mean(),
        "volatility_daily": return_df.std(),
        "win_rate": (return_df > 0).mean(),
        "best_day": return_df.max(),
        "worst_day": return_df.min(),
    })

    metrics["sharpe_ratio"] = (
        return_df.mean()
        / return_df.std()
        * np.sqrt(TRADING_DAYS)
    )

    # Downside deviation gemaess Sortino-Formel:
    # sqrt(1/N * sum(min(0, R_t - R_T)^2)) mit R_T = 0
    downside_part = return_df.clip(upper=0)
    downside_deviation = np.sqrt((downside_part ** 2).mean())

    metrics["sortino_ratio"] = (
        return_df.mean()
        / downside_deviation
        * np.sqrt(TRADING_DAYS)
    )

    years = (return_df.index[-1] - return_df.index[0]).days / 365.25
    cagr = np.exp(return_df.sum()) ** (1 / years) - 1

    running_max = equity_df.cummax()
    drawdowns = equity_df / running_max - 1
    max_drawdown = drawdowns.min()

    metrics["cagr"] = cagr
    metrics["max_drawdown"] = max_drawdown
    metrics["calmar_ratio"] = cagr / abs(max_drawdown)

    return metrics

In [ ]:
metrics_with_costs = calculate_metrics(
    comparison_returns_with_costs,
    equity_with_costs,
)

metrics_with_costs

In [ ]:
# Kosten-Szenarien testen
cost_scenarios_bps = [0, 1, 2.5, 5, 10, 25, 50]

cost_scenario_results = []

for cost_bps in cost_scenarios_bps:
    scenario_costs = turnover * (cost_bps / 10_000)
    scenario_returns = strategy_returns - scenario_costs
    scenario_equity = START_CAPITAL * np.exp(scenario_returns.cumsum())

    years = (scenario_returns.index[-1] - scenario_returns.index[0]).days / 365.25
    scenario_cagr = np.exp(scenario_returns.sum()) ** (1 / years) - 1

    scenario_total_return = scenario_equity.iloc[-1] / START_CAPITAL - 1

    cost_scenario_results.append({
        "cost_bps": cost_bps,
        "final_value": scenario_equity.iloc[-1],
        "total_return": scenario_total_return,
        "cagr": scenario_cagr,
    })

cost_scenario_table = pd.DataFrame(cost_scenario_results)

cost_scenario_table

In [ ]:
cost_scenario_table.plot(
    x="cost_bps",
    y="cagr",
    kind="bar",
    figsize=(10, 5),
    title=f"{asset}: Daily Reversal CAGR by Cost Assumption"
)

plt.axhline(0, color="black", linewidth=1)
plt.ylabel("CAGR")
plt.xlabel("Cost assumption in bps per turnover unit")
plt.grid(axis="y")
plt.show()

## 8. In-Sample / Out-of-Sample

Der Split zeigt, ob das Ergebnis über die Zeit stabil wirkt oder stark von einer Periode abhängt. Das ist kein endgültiger Beweis, aber eine notwendige Plausibilitätsprüfung.

In [ ]:
# In-Sample / Out-of-Sample Split
split_index = int(len(strategy_returns) * 0.75)

in_sample_returns = strategy_returns.iloc[:split_index]
out_sample_returns = strategy_returns.iloc[split_index:]

in_sample_buy_hold = asset_returns.iloc[:split_index]
out_sample_buy_hold = asset_returns.iloc[split_index:]

In [ ]:
split_results = []

for sample_name, bh_returns, rev_returns in [
    ("In-Sample", in_sample_buy_hold, in_sample_returns),
    ("Out-of-Sample", out_sample_buy_hold, out_sample_returns),
]:
    sample_returns = pd.DataFrame({
        BUY_HOLD_NAME: bh_returns,
        STRATEGY_NAME: rev_returns,
    })

    sample_equity = START_CAPITAL * np.exp(sample_returns.cumsum())

    sample_metrics = calculate_metrics(sample_returns, sample_equity)

    for strategy_name in sample_metrics.index:
        split_results.append({
            "sample": sample_name,
            "strategy": strategy_name,
            "cagr": sample_metrics.loc[strategy_name, "cagr"],
            "sharpe_ratio": sample_metrics.loc[strategy_name, "sharpe_ratio"],
            "max_drawdown": sample_metrics.loc[strategy_name, "max_drawdown"],
            "calmar_ratio": sample_metrics.loc[strategy_name, "calmar_ratio"],
            "win_rate": sample_metrics.loc[strategy_name, "win_rate"],
        })

split_results_table = pd.DataFrame(split_results)

split_results_table

## 9. Jahresperformance als Regime-Check

Die jährliche Auswertung zeigt, ob die Strategie in bestimmten Marktphasen hilft und in anderen Phasen klar zurückbleibt.

In [ ]:
# Jahresweise Returns
yearly_returns = pd.DataFrame({
    BUY_HOLD_NAME: asset_returns,
    STRATEGY_NAME: strategy_returns,
    f"{STRATEGY_NAME} ({COST_BPS} bps costs)": strategy_log_ret_after_costs,
})

yearly_performance = yearly_returns.groupby(yearly_returns.index.year).sum()
yearly_performance = np.exp(yearly_performance) - 1

yearly_performance


In [ ]:
yearly_performance.plot(
    kind="bar",
    figsize=(12, 6),
    title=f"{asset}: Yearly Performance"
)

plt.axhline(0, color="black", linewidth=1)
plt.ylabel("Return")
plt.xlabel("Year")
plt.grid(axis="y")
plt.show()

## 10. Cross-Asset-Vergleich

Zum Schluss testen wir dieselbe Daily-Reversal-Regel auf mehrere Asset-Klassen. Eine Strategie, die nur in einem Asset und nur vor Kosten funktioniert, ist keine robuste Universalregel.

In [ ]:
def run_daily_reversal_backtest(asset_name, return_data, cost_bps=10):
    asset_returns = return_data[asset_name].dropna()

    signal = -np.sign(asset_returns.shift(1))
    signal = signal.fillna(0)

    strategy_returns = signal * asset_returns

    turnover = signal.diff().abs()
    turnover.iloc[0] = abs(signal.iloc[0])

    costs = turnover * (cost_bps / 10_000)
    strategy_returns_after_costs = strategy_returns - costs

    comparison_returns = pd.DataFrame({
        BUY_HOLD_NAME: asset_returns,
        STRATEGY_NAME: strategy_returns,
        f"{STRATEGY_NAME} ({cost_bps} bps costs)": strategy_returns_after_costs,
    })

    equity_curves = START_CAPITAL * np.exp(comparison_returns.cumsum())

    metrics = calculate_metrics(comparison_returns, equity_curves)
    metrics["asset"] = asset_name
    metrics["final_value"] = equity_curves.iloc[-1]

    return comparison_returns, equity_curves, metrics

In [ ]:
all_metrics = []
all_equity_curves = {}

for asset_name in log_returns.columns:
    asset_comparison_returns, asset_equity_curves, asset_metrics = run_daily_reversal_backtest(
        asset_name=asset_name,
        return_data=log_returns,
        cost_bps=COST_BPS,
    )

    all_metrics.append(asset_metrics)
    all_equity_curves[asset_name] = asset_equity_curves

cross_asset_metrics = pd.concat(all_metrics)

cross_asset_metrics = cross_asset_metrics.reset_index().rename(columns={"index": "strategy"})

cross_asset_metrics

In [ ]:
cross_asset_summary = cross_asset_metrics[[
    "asset",
    "strategy",
    "cagr",
    "sharpe_ratio",
    "sortino_ratio",
    "max_drawdown",
    "calmar_ratio",
    "win_rate",
    "final_value",
]]

cross_asset_summary.sort_values(["asset", "strategy"])

In [ ]:
cross_asset_cagr = cross_asset_summary.pivot(
    index="asset",
    columns="strategy",
    values="cagr",
)

cross_asset_cagr

In [ ]:
cross_asset_cagr.plot(
    kind="bar",
    figsize=(12, 6),
    title=f"Daily Reversal Across Assets ({COST_BPS} bps costs)"
)

plt.axhline(0, color="black", linewidth=1)
plt.ylabel("CAGR")
plt.xlabel("Asset")
plt.grid(axis="y")
plt.show()

## 11. Fazit und nächste Research-Fragen

Die Daily-Reversal-Strategie sieht auf Ethereum im Brutto-Backtest zunächst interessant aus. Sie schlägt in diesem Sample Buy-and-Hold bei einigen Kennzahlen und reduziert den maximalen Drawdown leicht. Gleichzeitig bleibt der Drawdown extrem hoch.

Der entscheidende Stress-Test sind die Kosten. Schon moderate Kostenannahmen drehen das Ergebnis deutlich. Der Grund ist nicht mysteriös: Die Strategie hat hohen Turnover. Ein kleines Signal kann ökonomisch wertlos werden, wenn die Umsetzungskosten größer sind als der historische Bruttovorteil.

Der Jahresvergleich zeigt außerdem eine klare Regime-Abhängigkeit. In manchen Jahren hilft Daily Reversal, in starken Trendjahren bleibt sie deutlich hinter Buy-and-Hold zurück. Der Cross-Asset-Test bestätigt, dass diese Regel keine universelle Lösung ist: Für Gold, EUR/USD und den S&P-500-Proxy wirkt sie im getesteten Setup nicht robust.

Die wichtigste Lehre ist deshalb nicht: Diese Strategie blind handeln. Die wichtigste Lehre ist: Eine Strategieidee wird erst durch Benchmark, Kosten, Drawdown, Periodencheck und Cross-Asset-Test zu Research.

Offene nächste Research-Fragen:

- Wie verändert sich das Ergebnis bei Wochenreturns statt Tagesreturns?
- Gibt es niedrigere-Turnover-Varianten der Reversal-Idee?
- Funktioniert der Effekt eher in bestimmten Volatilitäts- oder Trendregimen?
- Wie sähe ein sauberer Test mit realistischeren Crypto-Futures-Kosten, Funding und Slippage aus?
- Wie unterscheidet sich diese Single-Asset-Time-Series-Regel von cross-sectional Crypto-Reversal-Ansätzen aus der Literatur?